In [1]:
import os, re, math
from pathlib import Path
from functools import lru_cache
from typing import Dict, Tuple, List

import numpy as np
import torch
from torch.utils.data import Dataset, DataLoader, random_split
from torch.cuda.amp import autocast, GradScaler

print("PyTorch:", torch.__version__, "| CUDA:", torch.version.cuda, "| GPUs:", torch.cuda.device_count())
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
device
POSITIONS_XY: Dict[int, Tuple[float, float]] = {
    0: (-113.97308,  -65.80239), 1: ( -75.98205,  -87.73651), 2: ( -37.99103, -109.67064),
    3: (   0.00000, -131.60477), 4: (-113.97308,  -21.93413), 5: ( -75.98205,  -43.86826),
    6: ( -37.99103,  -65.80239), 7: (   0.00000,  -87.73651), 8: (  37.99103, -109.67064),
    9: (-113.97308,   21.93413), 10: ( -75.98205,    0.00000), 11: ( -37.99103,  -21.93413),
    12: (   0.00000,  -43.86826), 13: (  37.99103,  -65.80239), 14: (  75.98205,  -87.73651),
    15: (-113.97308,   65.80239), 16: ( -75.98205,   43.86826), 17: ( -37.99103,   21.93413),
    18: (   0.00000,    0.00000), 19: (  37.99103,  -21.93413), 20: (  75.98205,  -43.86826),
    21: ( 113.97308,  -65.80239), 22: ( -75.98205,   87.73651), 23: ( -37.99103,   65.80239),
    24: (   0.00000,   43.86826), 25: (  37.99103,   21.93413), 26: (  75.98205,    0.00000),
    27: ( 113.97308,  -21.93413), 28: ( -37.99103,  109.67064), 29: (   0.00000,   87.73651),
    30: (  37.99103,   65.80239), 31: (  75.98205,   43.86826), 32: ( 113.97308,   21.93413),
    33: (   0.00000,  131.60477), 34: (  37.99103,  109.67064), 35: (  75.98205,   87.73651),
    36: ( 113.97308,   65.80239),
}



The history saving thread hit an unexpected error (DatabaseError('database disk image is malformed')).History will not be written to the database.
PyTorch: 2.9.0+cu126 | CUDA: 12.6 | GPUs: 4


In [2]:
# expects you already defined:
import zstandard as zstd

# def load_traces_from_zstd(input_path, n_traces, dtype=np.float16, trace_shape=(56, 300000)) -> np.ndarray
def load_traces_from_zstd(input_path, n_traces, dtype=np.float16, trace_shape=(56, 300000)) -> np.ndarray:
    """
    Load a list of numpy arrays (traces) from a compressed Zstandard (.zst) file and return a single stacked ndarray.
    """
    def unshuffle_bytes(data: bytes, dtype=np.float16, shape=(56, 300000)) -> np.ndarray:
        itemsize = np.dtype(dtype).itemsize
        num_elements = np.prod(shape)
        reshaped = np.frombuffer(data, dtype=np.uint8).reshape(itemsize, num_elements).T
        unshuffled = reshaped.reshape(-1)
        return unshuffled.view(dtype).reshape(shape)

    decompressor = zstd.ZstdDecompressor()
    with open(input_path, 'rb') as f:
        compressed_content = f.read()
        decompressed = decompressor.decompress(compressed_content)

    trace_size_bytes = np.prod(trace_shape) * np.dtype(dtype).itemsize
    expected_size = n_traces * trace_size_bytes
    if len(decompressed) != expected_size:
        raise ValueError("Decompressed size does not match expected size")

    traces = []
    for i in range(n_traces):
        start = i * trace_size_bytes
        end = start + trace_size_bytes
        trace_bytes = decompressed[start:end]
        trace = unshuffle_bytes(trace_bytes, dtype=dtype, shape=trace_shape)
        traces.append(trace)

    return np.stack(traces).astype(np.float32)
class ZstdTraceDataset(Dataset):
    """
    Each file traces_<idx>.zst contains (60, 56, 300000).
    We expose each repeat as a sample, and downsample time to seq_len_target (default 32768).
    Targets: normalized position (x/150, y/150, z_norm=0).
    """
    def __init__(self, data_dir: str, seq_len_target: int = 300000, repeats: int = 60):
        self.data_dir = Path(data_dir)
        self.seq_len_target = int(seq_len_target)
        self.repeats = int(repeats)

        self.files: List[Tuple[int, Path]] = []
        pat = re.compile(r"traces_(\d+)\.zst$")
        for p in sorted(self.data_dir.glob("traces_*.zst")):
            m = pat.search(p.name)
            if not m: 
                continue
            idx = int(m.group(1))
            if idx in POSITIONS_XY:
                self.files.append((idx, p))
        if not self.files:
            raise FileNotFoundError(f"No usable traces_*.zst in {self.data_dir}")

        # (file_idx, repeat_idx)
        self.index = [(fi, ri) for fi in range(len(self.files)) for ri in range(self.repeats)]

        # small cache (avoid re-decompressing the same file repeatedly)
        self._load_file_cached = lru_cache(maxsize=1)(self._load_file)

    def __len__(self): 
        return len(self.index)

    def _load_file(self, path_str: str) -> np.ndarray:
        return load_traces_from_zstd(path_str, n_traces=self.repeats, dtype=np.float16, trace_shape=(56, 300000))
        # shape: (60, 56, 300000) float32

    def _downsample(self, x_56_T: np.ndarray) -> np.ndarray:
        T = x_56_T.shape[-1]
        if T == self.seq_len_target:
            return x_56_T
        stride = T / self.seq_len_target
        idxs = (np.arange(self.seq_len_target) * stride).astype(np.int64)
        idxs = np.clip(idxs, 0, T - 1)
        return x_56_T[:, idxs]

    def __getitem__(self, i: int):
        fi, ri = self.index[i]
        idx, path = self.files[fi]

        # load file & pick repeat
        arr = self._load_file_cached(str(path))       # (60, 56, 300000)
        x_np = arr[ri]                                 # (56, 300000)
        x_np = self._downsample(x_np)                  # (56, seq_len_target)

        # labels (normalized)
        x_mm, y_mm = POSITIONS_XY[idx]
        pos = np.array([x_mm/150.0, y_mm/150.0, 0.0], dtype=np.float32)

        x = torch.from_numpy(x_np)     # (56, T) float32
        y = torch.from_numpy(pos)      # (3,)    float32
        return x, y


In [3]:
DATA_DIR = "/ceph/dwong/trigger_samples/v2_300k_xy_50ev/"  # <-- change me
ds = ZstdTraceDataset(DATA_DIR, seq_len_target=300000, repeats=60)
len(ds), ds[0][0].shape, ds[0][1]


(2220, torch.Size([56, 300000]), tensor([-0.7598, -0.4387,  0.0000]))

In [4]:
from ICL_toymodel import ICLConfig, PR  # import from your file/module

cfg = ICLConfig(
    n_channels=56,
    seq_len=300000,
    ch_chunk=8,          # safe on 12GB cards
    t_chunk=None,        # set to e.g. 4096 if you try very long seqs
    tfm_checkpoint=False # set True to save VRAM if needed
)

model = PR(cfg).to(device)
opt = torch.optim.AdamW(model.parameters(), lr=3e-4, weight_decay=1e-4)
scaler = GradScaler(enabled=True)

train_loader = DataLoader(ds, batch_size=4, shuffle=True, num_workers=2, pin_memory=True, drop_last=True)

print("Batches per epoch:", len(train_loader))


/home/dwong/anaconda3/envs/icl/lib/python3.12/site-packages/torch/nn/modules/transformer.py:392: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.self_attn.batch_first was not True(use batch_first for better inference performance)
  warnings.warn(


Batches per epoch: 555


/tmp/ipykernel_3324463/3014184769.py:13: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = GradScaler(enabled=True)


In [5]:
import torch
# 1) turn off cuDNN so LSTM uses the native kernel (works fine on TITAN X)
torch.backends.cudnn.enabled = False

# (optional perf flags — harmless)
torch.backends.cuda.matmul.allow_tf32 = True
torch.backends.cudnn.allow_tf32 = True
model.train()
for epoch in range(2):  # quick test
    running = 0.0
    for xb, pos in train_loader:
        # make sure inputs are contiguous float32 (cuDNN off anyway, but good hygiene)
        xb  = xb.to(device, non_blocking=True).contiguous().float()   # (B, 56, T)
        pos = pos.to(device, non_blocking=True).contiguous().float()  # (B, 3)

        opt.zero_grad(set_to_none=True)

        # IMPORTANT: no autocast here; keep fp32 for the LSTM
        out  = model(xb)                               # {"pos": (B,3)}
        loss = torch.nn.functional.mse_loss(out["pos"], pos)

        loss.backward()
        opt.step()

        running += loss.item() * xb.size(0)

    print(f"epoch {epoch+1}: train MSE(norm) = {running / len(ds):.6f}")


/home/dwong/anaconda3/envs/icl/lib/python3.12/site-packages/torch/backends/__init__.py:46: UserWarning: Please use the new API settings to control TF32 behavior, such as torch.backends.cudnn.conv.fp32_precision = 'tf32' or torch.backends.cuda.matmul.fp32_precision = 'ieee'. Old settings, e.g, torch.backends.cuda.matmul.allow_tf32 = True, torch.backends.cudnn.allow_tf32 = True, allowTF32CuDNN() and allowTF32CuBLAS() will be deprecated after Pytorch 2.9. Please see https://pytorch.org/docs/main/notes/cuda.html#tensorfloat-32-tf32-on-ampere-and-later-devices (Triggered internally at /pytorch/aten/src/ATen/Context.cpp:80.)
  self.setter(val)


OutOfMemoryError: CUDA out of memory. Tried to allocate 2.00 MiB. GPU 0 has a total capacity of 11.92 GiB of which 0 bytes is free. Including non-PyTorch memory, this process has 11.92 GiB memory in use. Of the allocated memory 11.32 GiB is allocated by PyTorch, and 13.61 MiB is reserved by PyTorch but unallocated. If reserved but unallocated memory is large try setting PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True to avoid fragmentation.  See documentation for Memory Management  (https://pytorch.org/docs/stable/notes/cuda.html#environment-variables)